# Exploitation Zone
Secuencia de la fase:
1. Configuracion de acceso a trusted y persistencia en exploitation.
2. Utilidades numericas, parsing y lectura de DuckDB.
3. Agregacion de clima y perfiles territoriales.
4. Enriquecimiento de listings Airbnb.
5. Construccion de vistas analiticas por zona y por zona-dia.
6. Materializacion de la base unificada final.


In [1]:
import json
import statistics
from collections import Counter, defaultdict
from contextlib import nullcontext
from pathlib import Path
from typing import Iterable

import duckdb

ROOT = Path.cwd()
TRUSTED_DB = ROOT / "trusted_zone" / "trusted_zone.duckdb"
EXPLOITATION_ZONE = ROOT / "exploitation_zone"
EXPLOITATION_DB = EXPLOITATION_ZONE / "exploitation_zone.duckdb"
EXPLOITATION_REPORTS = EXPLOITATION_ZONE / "reports"
CORE_ZONE_TABLES = [
    "pics",
    "hotels",
    "weather",
    "airbnb_listings",
    "airbnb_neighbourhoods",
    "airbnb_reviews",
    "income_2022",
    "hut_licenses",
]


def ensure_phase_dirs() -> None:
    EXPLOITATION_REPORTS.mkdir(parents=True, exist_ok=True)


In [2]:
def infer_duckdb_type(values: Iterable[object]) -> str:
    seen = {type(value) for value in values if value not in ("", None)}
    if not seen:
        return "TEXT"
    if seen.issubset({int}):
        return "BIGINT"
    if seen.issubset({int, float}):
        return "DOUBLE"
    return "TEXT"

def write_duckdb_table(db_path: Path, table_name: str, rows: list[dict[str, object]]) -> None:
    if not rows:
        return
    if duckdb is None:
        raise RuntimeError("DuckDB no está instalado.")
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = duckdb.connect(str(db_path))
    try:
        columns = list(rows[0].keys())
        types = {col: infer_duckdb_type(row.get(col) for row in rows) for col in columns}
        conn.execute(f'DROP TABLE IF EXISTS "{table_name}"')
        conn.execute(f'CREATE TABLE "{table_name}" ({", ".join(f"""\"{col}\" {types[col]}""" for col in columns)})')
        placeholders = ", ".join("?" for _ in columns)
        sql = f'INSERT INTO "{table_name}" ({", ".join(f"""\"{c}\"""" for c in columns)}) VALUES ({placeholders})'
        conn.executemany(sql, [[row.get(col) for col in columns] for row in rows])
        conn.commit()
    finally:
        conn.close()

def read_duckdb_rows(db_path: Path, table_name: str, order_by: str | None = None) -> list[dict[str, object]]:
    if duckdb is None:
        raise RuntimeError("DuckDB no está instalado.")
    conn = duckdb.connect(str(db_path), read_only=True)
    try:
        query = f'SELECT * FROM "{table_name}"'
        if order_by:
            query += f" ORDER BY {order_by}"
        cursor = conn.execute(query)
        columns = [description[0] for description in cursor.description]
        return [dict(zip(columns, row)) for row in cursor.fetchall()]
    finally:
        conn.close()

def load_tables_from_duckdb(db_path: Path, table_names: list[str]) -> dict[str, list[dict[str, object]]]:
    return {table_name: read_duckdb_rows(db_path, table_name) for table_name in table_names}

def write_json(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

def strip_text(value: str | None) -> str:
    if value is None:
        return ""
    return str(value).replace("\ufeff", "").strip()

def parse_float(value: str | None) -> float | None:
    text = strip_text(value)
    if not text:
        return None
    try:
        return float(text.replace(",", "."))
    except ValueError:
        return None

def parse_int(value: str | None) -> int | None:
    number = parse_float(value)
    return int(number) if number is not None else None

def parse_bool(value: str | None) -> int | None:
    text = strip_text(value).lower()
    if not text:
        return None
    if text in {"true", "1", "yes", "y", "t"}:
        return 1
    if text in {"false", "0", "no", "n", "f"}:
        return 0
    return None

def parse_price(value: str | None) -> float | None:
    text = strip_text(value)
    if not text:
        return None
    cleaned = text.replace("$", "").replace("€", "").replace(",", "").strip()
    try:
        return float(cleaned)
    except ValueError:
        return None

def mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0

def median(values: list[float]) -> float:
    return float(statistics.median(values)) if values else 0.0


In [3]:
def aggregate_weather_daily(weather_rows: list[dict[str, object]]) -> list[dict[str, object]]:
    grouped: dict[str, list[dict[str, object]]] = defaultdict(list)
    for row in weather_rows:
        grouped[str(row["date"])].append(row)
    daily_rows = []
    for date in sorted(grouped):
        rows = grouped[date]
        daily_rows.append(
            {
                "date": date,
                "temperature_avg": round(sum(r["temperature_avg"] for r in rows if r["temperature_avg"] is not None) / len([r for r in rows if r["temperature_avg"] is not None]), 4),
                "humidity_avg": round(sum(r["humidity_avg"] for r in rows if r["humidity_avg"] is not None) / len([r for r in rows if r["humidity_avg"] is not None]), 4),
                "pressure_avg": round(sum(r["pressure_avg"] for r in rows if r["pressure_avg"] is not None) / len([r for r in rows if r["pressure_avg"] is not None]), 4),
                "radiation_avg": round(sum(r["radiation_avg"] for r in rows if r["radiation_avg"] is not None) / len([r for r in rows if r["radiation_avg"] is not None]), 4),
                "wind_speed_avg": round(sum(r["wind_speed_avg"] for r in rows if r["wind_speed_avg"] is not None) / len([r for r in rows if r["wind_speed_avg"] is not None]), 4),
                "rain_total": round(sum(r["rain_total"] or 0 for r in rows), 4),
                "records_per_day": len(rows),
            }
        )
    return daily_rows

def build_district_profile(trusted_rows: dict[str, list[dict[str, object]]]) -> list[dict[str, object]]:
    hotel_by_district = defaultdict(list)
    pic_by_district = defaultdict(list)
    for row in trusted_rows["hotels"]:
        hotel_by_district[str(row["district_name"])].append(row)
    for row in trusted_rows["pics"]:
        pic_by_district[str(row["district_name"])].append(row)
    profile = []
    for district in sorted(set(hotel_by_district) | set(pic_by_district)):
        hotels = hotel_by_district[district]
        pics = pic_by_district[district]
        categories = Counter(row["category_name"] for row in hotels if row["category_name"])
        profile.append(
            {
                "district_name": district,
                "hotels_count": len(hotels),
                "pics_count": len(pics),
                "neighborhood_count": len({row["neighborhood_name"] for row in hotels + pics if row["neighborhood_name"]}),
                "top_hotel_category": categories.most_common(1)[0][0] if categories else "",
                "tourism_asset_score": len(hotels) * 2 + len(pics),
            }
        )
    return profile

def build_neighborhood_tourism_profile(trusted_rows: dict[str, list[dict[str, object]]]) -> list[dict[str, object]]:
    pics_by_neigh, hotels_by_neigh, district_by_neigh = defaultdict(list), defaultdict(list), {}
    for row in trusted_rows["pics"]:
        name = str(row["neighborhood_name"])
        pics_by_neigh[name].append(row)
        district_by_neigh[name] = str(row["district_name"])
    for row in trusted_rows["hotels"]:
        name = str(row["neighborhood_name"])
        hotels_by_neigh[name].append(row)
        district_by_neigh[name] = str(row["district_name"])
    for row in trusted_rows.get("airbnb_neighbourhoods", []):
        district_by_neigh[str(row["neighborhood_name"])] = str(row["district_name"])
    profile = []
    for name in sorted(set(district_by_neigh) | set(pics_by_neigh) | set(hotels_by_neigh)):
        pics = pics_by_neigh[name]
        hotels = hotels_by_neigh[name]
        profile.append(
            {
                "district_name": district_by_neigh.get(name, ""),
                "neighborhood_name": name,
                "pics_count": len(pics),
                "hotels_count": len(hotels),
                "tourism_asset_score": len(hotels) * 2 + len(pics),
            }
        )
    return profile

def build_district_day_features(district_profile: list[dict[str, object]], weather_daily: list[dict[str, object]]) -> list[dict[str, object]]:
    rows = []
    for district in district_profile:
        for weather in weather_daily:
            rows.append(
                {
                    "date": weather["date"],
                    "district_name": district["district_name"],
                    "hotels_count": district["hotels_count"],
                    "pics_count": district["pics_count"],
                    "neighborhood_count": district["neighborhood_count"],
                    "tourism_asset_score": district["tourism_asset_score"],
                    "temperature_avg": weather["temperature_avg"],
                    "humidity_avg": weather["humidity_avg"],
                    "pressure_avg": weather["pressure_avg"],
                    "radiation_avg": weather["radiation_avg"],
                    "wind_speed_avg": weather["wind_speed_avg"],
                    "rain_total": weather["rain_total"],
                }
            )
    return rows


In [4]:
def build_income_profile(trusted_rows: dict[str, list[dict[str, object]]]) -> tuple[list[dict[str, object]], list[dict[str, object]]]:
    district_grouped: dict[str, list[dict[str, object]]] = defaultdict(list)
    neighborhood_grouped: dict[tuple[str, str], list[dict[str, object]]] = defaultdict(list)
    for row in trusted_rows.get("income_2022", []):
        district_grouped[str(row["district_name"])].append(row)
        neighborhood_grouped[(str(row["district_name"]), str(row["neighborhood_name"]))].append(row)
    district_rows = []
    for district, rows in sorted(district_grouped.items()):
        incomes = [float(r["income_eur"]) for r in rows if r.get("income_eur") is not None]
        district_rows.append(
            {
                "district_name": district,
                "avg_income_eur": round(mean(incomes), 4) if incomes else 0.0,
                "median_income_eur": round(median(incomes), 4) if incomes else 0.0,
                "census_section_count": len(rows),
            }
        )
    neighborhood_rows = []
    for (district, neighborhood), rows in sorted(neighborhood_grouped.items()):
        incomes = [float(r["income_eur"]) for r in rows if r.get("income_eur") is not None]
        neighborhood_rows.append(
            {
                "district_name": district,
                "neighborhood_name": neighborhood,
                "avg_income_eur": round(mean(incomes), 4) if incomes else 0.0,
                "median_income_eur": round(median(incomes), 4) if incomes else 0.0,
                "census_section_count": len(rows),
            }
        )
    return district_rows, neighborhood_rows


def build_hut_profile(trusted_rows: dict[str, list[dict[str, object]]]) -> tuple[list[dict[str, object]], list[dict[str, object]]]:
    district_grouped: dict[str, list[dict[str, object]]] = defaultdict(list)
    neighborhood_grouped: dict[tuple[str, str], list[dict[str, object]]] = defaultdict(list)
    for row in trusted_rows.get("hut_licenses", []):
        district_grouped[str(row["district_name"])].append(row)
        neighborhood_grouped[(str(row["district_name"]), str(row["neighborhood_name"]))].append(row)
    district_rows = []
    for district, rows in sorted(district_grouped.items()):
        beds = [float(r["licensed_beds"]) for r in rows if r.get("licensed_beds") is not None]
        district_rows.append(
            {
                "district_name": district,
                "hut_license_count": len(rows),
                "avg_hut_beds": round(mean(beds), 4) if beds else 0.0,
                "total_hut_beds": round(sum(beds), 4) if beds else 0.0,
            }
        )
    neighborhood_rows = []
    for (district, neighborhood), rows in sorted(neighborhood_grouped.items()):
        beds = [float(r["licensed_beds"]) for r in rows if r.get("licensed_beds") is not None]
        neighborhood_rows.append(
            {
                "district_name": district,
                "neighborhood_name": neighborhood,
                "hut_license_count": len(rows),
                "avg_hut_beds": round(mean(beds), 4) if beds else 0.0,
                "total_hut_beds": round(sum(beds), 4) if beds else 0.0,
            }
        )
    return district_rows, neighborhood_rows


In [5]:
def build_airbnb_listing_enriched(trusted_rows: dict[str, list[dict[str, object]]], district_profile: list[dict[str, object]], neighborhood_profile: list[dict[str, object]]) -> list[dict[str, object]]:
    district_map = {row["district_name"]: row for row in district_profile}
    neighborhood_map = {row["neighborhood_name"]: row for row in neighborhood_profile}
    review_map = {row["listing_id"]: row for row in trusted_rows.get("airbnb_reviews", [])}
    enriched = []
    for row in trusted_rows.get("airbnb_listings", []):
        review = review_map.get(row["listing_id"], {})
        neigh = neighborhood_map.get(row["neighborhood_name"], {})
        district = district_map.get(row["district_name"], {})
        enriched.append(
            {
                **row,
                "review_count_from_reviews_csv": review.get("review_count", 0),
                "avg_comment_length": review.get("avg_comment_length", 0.0),
                "neighborhood_pics_count": neigh.get("pics_count", 0),
                "neighborhood_hotels_count": neigh.get("hotels_count", 0),
                "neighborhood_tourism_asset_score": neigh.get("tourism_asset_score", 0),
                "district_pics_count": district.get("pics_count", 0),
                "district_hotels_count": district.get("hotels_count", 0),
                "district_tourism_asset_score": district.get("tourism_asset_score", 0),
            }
        )
    return enriched

def build_airbnb_zone_features(listing_rows: list[dict[str, object]]) -> list[dict[str, object]]:
    grouped: dict[tuple[str, str], list[dict[str, object]]] = defaultdict(list)
    for row in listing_rows:
        grouped[(str(row["district_name"]), str(row["neighborhood_name"]))].append(row)
    features = []
    for (district, neighborhood), rows in sorted(grouped.items()):
        prices = [float(r["price"]) for r in rows if r["price"] is not None]
        ratings = [float(r["review_scores_rating"]) for r in rows if r["review_scores_rating"] is not None]
        accommodates = [float(r["accommodates"]) for r in rows if r["accommodates"] is not None]
        bedrooms = [float(r["bedrooms"]) for r in rows if r["bedrooms"] is not None]
        availabilities = [float(r["availability_365"]) for r in rows if r["availability_365"] is not None]
        reviews_per_month = [float(r["reviews_per_month"]) for r in rows if r["reviews_per_month"] is not None]
        rows_count = len(rows)
        features.append(
            {
                "district_name": district,
                "neighborhood_name": neighborhood,
                "listing_count": rows_count,
                "avg_price": round(mean(prices), 4),
                "median_price": round(median(prices), 4),
                "avg_rating": round(mean(ratings), 4) if ratings else 0.0,
                "avg_accommodates": round(mean(accommodates), 4) if accommodates else 0.0,
                "avg_bedrooms": round(mean(bedrooms), 4) if bedrooms else 0.0,
                "avg_availability_365": round(mean(availabilities), 4) if availabilities else 0.0,
                "avg_reviews_per_month": round(mean(reviews_per_month), 4) if reviews_per_month else 0.0,
                "entire_home_ratio": round(sum(1 for r in rows if r["room_type"] == "Entire home/apt") / rows_count, 4),
                "private_room_ratio": round(sum(1 for r in rows if r["room_type"] == "Private room") / rows_count, 4),
                "superhost_ratio": round(sum(1 for r in rows if r["host_is_superhost"] == 1) / rows_count, 4),
                "instant_bookable_ratio": round(sum(1 for r in rows if r["instant_bookable"] == 1) / rows_count, 4),
                "avg_review_count": round(mean([float(r["review_count_from_reviews_csv"]) for r in rows]), 4),
                "avg_comment_length": round(mean([float(r["avg_comment_length"]) for r in rows]), 4),
                "neighborhood_pics_count": rows[0]["neighborhood_pics_count"],
                "neighborhood_hotels_count": rows[0]["neighborhood_hotels_count"],
                "neighborhood_tourism_asset_score": rows[0]["neighborhood_tourism_asset_score"],
                "district_tourism_asset_score": rows[0]["district_tourism_asset_score"],
            }
        )
    return features

def build_airbnb_zone_day_features(airbnb_listing_enriched: list[dict[str, object]], weather_daily: list[dict[str, object]]) -> list[dict[str, object]]:
    listing_map = {str(row["listing_id"]): row for row in airbnb_listing_enriched}
    weather_map = {str(row["date"]): row for row in weather_daily}
    grouped: dict[tuple[str, str, str], dict[str, float | int | str]] = {}
    calendar_rows = read_duckdb_rows(TRUSTED_DB, "airbnb_calendar", order_by="date, listing_id")
    context = nullcontext()
    with context:
        for row in calendar_rows:
            listing = listing_map.get(strip_text(row.get("listing_id")))
            if not listing:
                continue
            date = strip_text(row.get("date"))
            key = (str(listing["district_name"]), str(listing["neighborhood_name"]), date)
            bucket = grouped.setdefault(
                key,
                {
                    "district_name": str(listing["district_name"]),
                    "neighborhood_name": str(listing["neighborhood_name"]),
                    "date": date,
                    "listing_count": 0,
                    "available_count": 0,
                    "booked_count": 0,
                    "price_sum": 0.0,
                    "price_count": 0,
                    "minimum_nights_sum": 0.0,
                    "minimum_nights_count": 0,
                    "neighborhood_tourism_asset_score": float(listing["neighborhood_tourism_asset_score"]),
                    "district_tourism_asset_score": float(listing["district_tourism_asset_score"]),
                },
            )
            bucket["listing_count"] = int(bucket["listing_count"]) + 1
            available = parse_bool(row.get("available"))
            if available == 1:
                bucket["available_count"] = int(bucket["available_count"]) + 1
            elif available == 0:
                bucket["booked_count"] = int(bucket["booked_count"]) + 1
            row_price = parse_price(row.get("adjusted_price")) or parse_price(row.get("price")) or (float(listing["price"]) if listing["price"] is not None else None)
            if row_price is not None:
                bucket["price_sum"] = float(bucket["price_sum"]) + row_price
                bucket["price_count"] = int(bucket["price_count"]) + 1
            row_min = parse_int(row.get("minimum_nights"))
            if row_min is not None:
                bucket["minimum_nights_sum"] = float(bucket["minimum_nights_sum"]) + row_min
                bucket["minimum_nights_count"] = int(bucket["minimum_nights_count"]) + 1
    final_rows = []
    for key in sorted(grouped):
        row = grouped[key]
        count = int(row["listing_count"])
        weather = weather_map.get(str(row["date"]), {})
        final_rows.append(
            {
                "district_name": row["district_name"],
                "neighborhood_name": row["neighborhood_name"],
                "date": row["date"],
                "listing_count": count,
                "available_count": int(row["available_count"]),
                "booked_count": int(row["booked_count"]),
                "availability_rate": round(int(row["available_count"]) / count, 4) if count else 0.0,
                "booked_rate": round(int(row["booked_count"]) / count, 4) if count else 0.0,
                "avg_calendar_price": round(float(row["price_sum"]) / int(row["price_count"]), 4) if int(row["price_count"]) else 0.0,
                "avg_minimum_nights": round(float(row["minimum_nights_sum"]) / int(row["minimum_nights_count"]), 4) if int(row["minimum_nights_count"]) else 0.0,
                "neighborhood_tourism_asset_score": row["neighborhood_tourism_asset_score"],
                "district_tourism_asset_score": row["district_tourism_asset_score"],
                "temperature_avg": weather.get("temperature_avg", 0.0),
                "humidity_avg": weather.get("humidity_avg", 0.0),
                "pressure_avg": weather.get("pressure_avg", 0.0),
                "radiation_avg": weather.get("radiation_avg", 0.0),
                "wind_speed_avg": weather.get("wind_speed_avg", 0.0),
                "rain_total": weather.get("rain_total", 0.0),
            }
        )
    return final_rows


In [6]:
ensure_phase_dirs()
trusted_rows = load_tables_from_duckdb(TRUSTED_DB, CORE_ZONE_TABLES)
district_income_profile, neighborhood_income_profile = build_income_profile(trusted_rows)
district_hut_profile, neighborhood_hut_profile = build_hut_profile(trusted_rows)
district_profile = build_district_profile(trusted_rows)
neighborhood_profile = build_neighborhood_tourism_profile(trusted_rows)
weather_daily = aggregate_weather_daily(trusted_rows["weather"])
district_day_features = build_district_day_features(district_profile, weather_daily)
airbnb_listing_enriched = build_airbnb_listing_enriched(trusted_rows, district_profile, neighborhood_profile)
airbnb_zone_features = build_airbnb_zone_features(airbnb_listing_enriched)
airbnb_zone_day_features = build_airbnb_zone_day_features(airbnb_listing_enriched, weather_daily)
outputs = {
    "district_profile": district_profile,
    "neighborhood_profile": neighborhood_profile,
    "district_income_profile": district_income_profile,
    "neighborhood_income_profile": neighborhood_income_profile,
    "district_hut_profile": district_hut_profile,
    "neighborhood_hut_profile": neighborhood_hut_profile,
    "weather_daily": weather_daily,
    "district_day_features": district_day_features,
    "airbnb_listing_enriched": airbnb_listing_enriched,
    "airbnb_zone_features": airbnb_zone_features,
    "airbnb_zone_day_features": airbnb_zone_day_features,
}
for dataset, rows in outputs.items():
    write_duckdb_table(EXPLOITATION_DB, dataset, rows)
write_json(EXPLOITATION_REPORTS / "exploitation_zone_report.json", {name: len(rows) for name, rows in outputs.items()})
{name: len(rows) for name, rows in outputs.items()}


{'district_profile': 10, 'neighborhood_profile': 74, 'district_income_profile': 10, 'neighborhood_income_profile': 73, 'district_hut_profile': 10, 'neighborhood_hut_profile': 65, 'weather_daily': 290, 'district_day_features': 2900, 'airbnb_listing_enriched': 15276, 'airbnb_zone_features': 71, 'airbnb_zone_day_features': 25954}